# View-18 interior comparison: Ground Truth vs Objective 1 vs DINO union-safe

This notebook visualizes the copied `dino_transplant_view18` results. The new method is **`union_safe`**, the strongest evaluated DINO hybrid: it keeps the Objective-1 exterior and adds only aligned retrieved voxels that fall inside its conservative enclosed volume.

Each comparison uses the same ground-truth-derived cut plane and camera:

- **Top row:** full voxel geometry with the camera-facing half removed.
- **Bottom row:** margin-2 internals. Ground truth is gold; prediction panels show **green = matched**, **red = extra**, and **blue = missing** internal voxels.

The saved visualization bundle contains three examples per category (12 total). Change `SAMPLE_INDEX` in the sample-browser cell to browse them.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pyvista as pv
from IPython.display import display
from PIL import Image

RESULTS_DIR = Path("results/dino_transplant_view18")
VIS_DIR = RESULTS_DIR / "visualizations"
RESOLUTION = 64
METRIC_MARGIN = 2
POINT_SIZE = 7

# Remove the camera-facing half along this direction. Try y or z if useful.
CUT_NORMAL = np.array([1.0, 0.0, 0.0])
CUT_FRACTION = 0.5

METHODS = [
    ("Ground truth", "ground_truth"),
    ("Objective 1 (view 18)", "objective1"),
    ("DINO union-safe", "union_safe"),
]
PREDICTION_METHODS = ("objective1", "union_safe")

if not VIS_DIR.is_dir():
    raise FileNotFoundError(f"Missing copied visualizations: {VIS_DIR}")

sample_ids = sorted(path.name for path in VIS_DIR.iterdir() if path.is_dir())
if not sample_ids:
    raise ValueError(f"No visualization samples found in {VIS_DIR}")

metrics = {}
with (RESULTS_DIR / "per_sample.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if (
            row["sample_id"] in sample_ids
            and row["method"] in PREDICTION_METHODS
            and int(row["margin"]) == METRIC_MARGIN
        ):
            metrics.setdefault(row["sample_id"], {})[row["method"]] = {
                "internal_f1": float(row["internal_f1"]),
                "internal_precision": float(row["internal_precision"]),
                "internal_recall": float(row["internal_recall"]),
                "exterior_iou": float(row["exterior_iou"]),
                "internal_ratio": float(row["pred_to_gt_internal_ratio"]),
                "retrieved_id": row["retrieved_id"],
            }

for sample_id in sample_ids:
    if set(metrics.get(sample_id, {})) != set(PREDICTION_METHODS):
        raise ValueError(f"Missing paired margin-{METRIC_MARGIN} metrics for {sample_id}")
    for _, filename in METHODS:
        path = VIS_DIR / sample_id / f"{filename}.ply"
        if not path.is_file():
            raise FileNotFoundError(f"Missing visualization PLY: {path}")

def f1_delta(sample_id):
    return (
        metrics[sample_id]["union_safe"]["internal_f1"]
        - metrics[sample_id]["objective1"]["internal_f1"]
    )

sample_ids.sort(key=f1_delta, reverse=True)

summary = {}
with (RESULTS_DIR / "summary.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if row["method"] in PREDICTION_METHODS and int(row["margin"]) == METRIC_MARGIN:
            summary[row["method"]] = row

objective1_mean = float(summary["objective1"]["internal_f1"])
union_mean = float(summary["union_safe"]["internal_f1"])
print(f"Verified {len(sample_ids)} visualization bundles from a 195-object view-18 evaluation")
print(f"Mean margin-{METRIC_MARGIN} internal F1: {objective1_mean:.4f} -> {union_mean:.4f} (delta {union_mean - objective1_mean:+.4f})")
print("Sample indices below are ordered from largest to smallest F1 change:")
for index, sample_id in enumerate(sample_ids):
    print(f"  {index:02d}: {sample_id:58s} delta {f1_delta(sample_id):+.3f}")


In [ ]:
def read_voxels(path):
    points = np.asarray(pv.read(path).points, dtype=np.float32).reshape(-1, 3)
    if len(points) == 0:
        return set()
    coordinates = np.floor((points + 0.5) * RESOLUTION).astype(np.int32)
    coordinates = np.clip(coordinates, 0, RESOLUTION - 1)
    return {tuple(voxel) for voxel in coordinates.tolist()}


def voxel_points(voxels):
    if not voxels:
        return np.empty((0, 3), dtype=np.float32)
    coordinates = np.asarray(sorted(voxels), dtype=np.float32)
    return (coordinates + 0.5) / RESOLUTION - 0.5


def interior(voxels, margin=2):
    internal = set(voxels)
    for axis in range(3):
        other_axes = [index for index in range(3) if index != axis]
        groups = {}
        for voxel in voxels:
            key = (voxel[other_axes[0]], voxel[other_axes[1]])
            groups.setdefault(key, []).append(voxel)
        for group in groups.values():
            minimum = min(voxel[axis] for voxel in group)
            maximum = max(voxel[axis] for voxel in group)
            for voxel in group:
                if voxel[axis] - minimum < margin or maximum - voxel[axis] < margin:
                    internal.discard(voxel)
    return internal


def camera_for_cut(normal, center, object_size):
    up = np.array([0.0, 0.0, 1.0])
    if abs(normal @ up) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    side = np.cross(up, normal)
    position = center + object_size * (2.0 * normal + 0.65 * side + 0.45 * up)
    return [position.tolist(), center.tolist(), up.tolist()]


def add_points(plotter, voxels, color):
    points = voxel_points(voxels)
    if len(points):
        plotter.add_points(
            points,
            color=color,
            point_size=POINT_SIZE,
            render_points_as_spheres=True,
        )


def render_comparison(sample_id):
    sample_dir = VIS_DIR / sample_id
    voxel_sets = {
        label: read_voxels(sample_dir / f"{filename}.ply")
        for label, filename in METHODS
    }

    ground_truth = voxel_sets["Ground truth"]
    gt_points = voxel_points(ground_truth)
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    projection = gt_points @ normal
    cut_offset = projection.max() - CUT_FRACTION * np.ptp(projection)
    center = (gt_points.min(axis=0) + gt_points.max(axis=0)) / 2.0
    object_size = np.ptp(gt_points, axis=0).max()

    plotter = pv.Plotter(shape=(2, 3), off_screen=True, window_size=(1800, 1100))
    background = "#0b1020"
    full_colors = {
        "Ground truth": "#d9d9d9",
        "Objective 1 (view 18)": "#f4a261",
        "DINO union-safe": "#57c7ff",
    }

    # Top row: the same cutaway plane for all three voxel sets.
    for column, (label, _) in enumerate(METHODS):
        visible = {
            voxel for voxel in voxel_sets[label]
            if np.asarray(voxel_points({voxel})[0]) @ normal <= cut_offset
        }
        plotter.subplot(0, column)
        plotter.set_background(background)
        add_points(plotter, visible, full_colors[label])
        plotter.add_text(f"{label} | cutaway", position="upper_left", color="white", font_size=11)

    # Bottom row: exact internal-voxel agreement used by the reported metric.
    gt_internal = interior(ground_truth, METRIC_MARGIN)
    for column, (label, _) in enumerate(METHODS):
        plotter.subplot(1, column)
        plotter.set_background(background)
        if label == "Ground truth":
            add_points(plotter, gt_internal, "#ffd166")
            plotter.add_text("Ground-truth internals", position="upper_left", color="white", font_size=11)
        else:
            predicted_internal = interior(voxel_sets[label], METRIC_MARGIN)
            add_points(plotter, predicted_internal & gt_internal, "#41d17d")
            add_points(plotter, predicted_internal - gt_internal, "#ff5a5f")
            add_points(plotter, gt_internal - predicted_internal, "#4d96ff")
            plotter.add_text(f"{label} | internal errors", position="upper_left", color="white", font_size=11)
            plotter.add_text("green matched | red extra | blue missing", position="lower_left", color="white", font_size=8)

    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, center, object_size)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.68 * object_size
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)


In [ ]:
SAMPLE_INDEX = 0  # Choose any index printed by the configuration cell.

sample_id = sample_ids[SAMPLE_INDEX]
objective1 = metrics[sample_id]["objective1"]
union_safe = metrics[sample_id]["union_safe"]
print(sample_id)
print(f"Retrieved training shape: {union_safe['retrieved_id']}")
print(
    f"Margin-{METRIC_MARGIN} internal F1: "
    f"{objective1['internal_f1']:.3f} -> {union_safe['internal_f1']:.3f} "
    f"(delta {f1_delta(sample_id):+.3f})"
)
print(
    f"Precision: {objective1['internal_precision']:.3f} -> {union_safe['internal_precision']:.3f} | "
    f"Recall: {objective1['internal_recall']:.3f} -> {union_safe['internal_recall']:.3f}"
)
print(
    f"Predicted/GT internal ratio: {objective1['internal_ratio']:.2f}x -> "
    f"{union_safe['internal_ratio']:.2f}x"
)
display(render_comparison(sample_id))


In [ ]:
SHOW_ALL = False  # Set True only when you want to render all 12 examples.

if SHOW_ALL:
    for index, sample_id in enumerate(sample_ids):
        objective1 = metrics[sample_id]["objective1"]
        union_safe = metrics[sample_id]["union_safe"]
        print(
            f"{index:02d}. {sample_id} | F1 "
            f"{objective1['internal_f1']:.3f} -> {union_safe['internal_f1']:.3f} "
            f"(delta {f1_delta(sample_id):+.3f})"
        )
        display(render_comparison(sample_id))
